In [8]:
here::i_am("rna_atac/dimensionality_reduction/WNN_MOFA.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(ArchR))

BPPARAM <- BiocParallel::bpparam()
BPPARAM$workers = 21

# Multi core using future - built in to seurat
plan("multicore", workers = 16)
options(future.globals.maxSize = 50 * 1024 ^ 3) # for 50 Gb RAM

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
      

In [2]:
args = list()
# MOFAsprintf('%s/results/rna_atac/combined_metadata.txt.gz',io$basedir)
args$metadata = file.path(io$basedir, 'results/rna_atac/combined_metadata.txt.gz')
args$MOFA = file.path(io$basedir, 'results/rna_atac/dimensionality_reduction/mofa/MOFA_factors.txt.gz')
args$MOFA_umap = file.path(io$basedir, 'results/rna_atac/dimensionality_reduction/mofa/MOFA_umap.txt.gz')

# RNA_sce
args$rna_sce = file.path(io$basedir, 'processed/rna/SingleCellExperiment.rds')

# Mapping Luke
args$integrated_object = '/rds/project/rds-SDzz0CATGms/users/ltgh2/projects/01_Eomes_invitro_HE_multiome/processed/rna/seurat_objects/all_anchors_20_rPCA.rds'

# outdir
args$outdir_clusters = file.path(io$basedir, 'results/rna_atac/dimensionality_reduction/')
dir.create(args$outdir_clusters, recursive=TRUE, showWarnings =FALSE)

args$outdir_markers = file.path(io$basedir, 'results/rna_atac/markers/')
dir.create(args$outdir_markers, recursive=TRUE, showWarnings =FALSE)

In [3]:
seurat = readRDS(sprintf('%s/seurat_object.rds', args$outdir_clusters))
markers = fread(sprintf('%s/markers_seurat.txt.gz', args$outdir_markers))
umap = fread(args$MOFA_umap)
meta = fread(args$metadata)

In [4]:
meta = as.data.table(seurat@meta.data, keep.rownames=T) %>% setnames('rn', 'cell')

In [9]:
args$archr_directory = file.path(io$basedir, 'processed/atac/archR')
ArchRProject <- loadArchRProject(args$archr_directory)[meta$cell]

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [10]:
addArchRThreads(1)

Setting default number of Parallel threads to 1.



In [11]:
ArchRProject <- addCellColData(ArchRProj = ArchRProject, 
                                    data = as.numeric(meta$seurat_clusters),
                                    cells = meta$cell,
                                    name = "seurat_clusters",
                                    force=TRUE)

In [12]:
out <- getMarkerFeatures( # suppressMessages(
          ArchRProj = ArchRProject, #.filt, 
          useMatrix = "PeakMatrix",
          groupBy = "seurat_clusters",
          testMethod = "wilcoxon", #"binomial",
          binarize=FALSE,
          bias = c("TSSEnrichment", "log10(nFrags)"),
          verbose=TRUE
        )

ArchR logging to : ArchRLogs/ArchR-getMarkerFeatures-c8fe56dc41542-Date-2023-02-22_Time-16-19-41.log
If there is an issue, please report to github with logFile!

MatrixClass = Sparse.Integer.Matrix

2023-02-22 16:19:46 : Matching Known Biases, 0.01 mins elapsed.

2023-02-22 16:20:24 : Computing Pairwise Tests (1 of 22), 0.65 mins elapsed.

Pairwise Test 1 : Seqnames chr1

Pairwise Test 1 : Seqnames chr10

Pairwise Test 1 : Seqnames chr11

Pairwise Test 1 : Seqnames chr12

Pairwise Test 1 : Seqnames chr13

Pairwise Test 1 : Seqnames chr14

Pairwise Test 1 : Seqnames chr15

Pairwise Test 1 : Seqnames chr16

Pairwise Test 1 : Seqnames chr17

Pairwise Test 1 : Seqnames chr18

Pairwise Test 1 : Seqnames chr19

Pairwise Test 1 : Seqnames chr2

Pairwise Test 1 : Seqnames chr3

Pairwise Test 1 : Seqnames chr4

Pairwise Test 1 : Seqnames chr5

Pairwise Test 1 : Seqnames chr6

Pairwise Test 1 : Seqnames chr7

Pairwise Test 1 : Seqnames chr8

Pairwise Test 1 : Seqnames chr9

Pairwise Test 1 : Seq